# Feature engineering

Builds the full gold feature table from silver transactions and inspects what each feature group adds: transaction, temporal, entity, and graph features.

**Prerequisites:** `make sample-data` and `make ingest`.

In [1]:
from transaction_risk.spark.session import create_spark_session_from_yaml
spark = create_spark_session_from_yaml('../conf/spark.local.yaml')


In [2]:
from transaction_risk.features.pipeline import build_feature_table
from transaction_risk.spark.io import read_table

transactions = read_table(spark, '../data/silver/transactions')
features = build_feature_table(transactions, feature_config_path='../conf/features.yaml')

added_columns = sorted(set(features.columns) - set(transactions.columns))
print(f'{len(added_columns)} feature columns added:')
for column in added_columns:
    print(' -', column)

38 feature columns added:
 - amount_log1p
 - destination_avg_amount
 - destination_balance_delta
 - destination_balance_is_zero
 - destination_balance_was_zero
 - destination_delta_minus_amount
 - destination_historical_fraud_count
 - destination_historical_fraud_rate
 - destination_in_degree
 - destination_max_amount
 - destination_total_tx_count
 - destination_unique_origins
 - edge_frequency
 - is_cash_in
 - is_cash_out
 - is_debit
 - is_merchant_destination
 - is_payment
 - is_transfer
 - large_amount_flag
 - origin_amount_mean_before
 - origin_amount_std_before
 - origin_amount_to_mean_ratio
 - origin_amount_zscore_before
 - origin_avg_amount
 - origin_balance_delta
 - origin_balance_is_zero
 - origin_balance_was_zero
 - origin_delta_minus_amount
 - origin_destination_avg_amount
 - origin_destination_pair_count
 - origin_max_amount
 - origin_out_degree
 - origin_total_tx_count
 - origin_tx_count_before
 - origin_unique_destinations
 - previous_step_by_origin
 - steps_since_previou

In [3]:
# Temporal features only look backwards in time: history columns are 0/-1 for an account's first transaction
features.select(
    'nameOrig',
    'step',
    'amount',
    'steps_since_previous_origin_tx',
    'origin_tx_count_before',
    'origin_amount_mean_before',
    'origin_amount_zscore_before',
).orderBy('nameOrig', 'step').show(10)

+--------+----+--------+------------------------------+----------------------+-------------------------+---------------------------+
|nameOrig|step|  amount|steps_since_previous_origin_tx|origin_tx_count_before|origin_amount_mean_before|origin_amount_zscore_before|
+--------+----+--------+------------------------------+----------------------+-------------------------+---------------------------+
|C0000001|  88| 7004.14|                            -1|                     0|                      0.0|                        0.0|
|C0000001| 144| 1427.39|                            56|                     1|                  7004.14|                        0.0|
|C0000001| 168| 6809.79|                            24|                     2|                 4215.765|         0.9302999058591472|
|C0000002| 103| 6130.05|                            -1|                     0|                      0.0|                        0.0|
|C0000002| 123| 1442.05|                            20|              

In [4]:
# The feature registry documents ownership, sources, and leakage risk for every feature
from transaction_risk.features.metadata import get_feature_registry

registry = get_feature_registry()
print(f'{len(registry)} registered features. Example entry:')
print(registry[0])
spark.stop()

55 registered features. Example entry:
FeatureDefinition(name='amount_log1p', group='transaction', dtype='double', description='Natural log of amount plus one.', source_columns=['amount'], leakage_risk='low', default_value=0, owner='risk-ml')
